# Banking Customer Analytics & Risk Engine — Modeling Experiments

This notebook demonstrates the supervised default risk classification workflow:
1. Feature selection and strict target leakage elimination
2. Stratified train/validation/test splitting (70% / 15% / 15%)
3. Scikit-learn `Pipeline` preprocessing and `LogisticRegression(class_weight='balanced')` fitting
4. Validation set decision threshold optimization
5. Held-out test evaluation against zero-skill baseline
6. Feature coefficient interpretability and odds ratios

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from banking_analytics.utils.config_loader import load_config
from banking_analytics.utils.io_utils import read_csv_safe
from banking_analytics.feature_engineer import select_features, split_data, build_preprocessor
from banking_analytics.classifier import (
    build_classification_pipeline,
    find_optimal_threshold,
    evaluate_classifier,
    extract_coefficients,
)
from sklearn.dummy import DummyClassifier

config = load_config("../config/config.yaml")
df = read_csv_safe(f"../{config['data']['processed_path']}")
print(f"Loaded clean dataset: {df.shape[0]} rows, {df.shape[1]} columns")

## 1. Feature Selection & Leakage Prevention Audit
Target and post-decision columns (`Risk_Score`, `Default_Risk`, `Approved_Amount`, `Interest_Rate`, `Application_ID`) are strictly excluded.

In [ ]:
X, y, num_cols, cat_cols, bin_cols = select_features(df)
print(f"Feature matrix X shape: {X.shape}")
print(f"Numeric features ({len(num_cols)}): {num_cols[:5]}...")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"Binary features ({len(bin_cols)}): {bin_cols}")

## 2. Stratified Data Split (Train=70%, Val=15%, Test=15%)

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y, train_size=0.70, val_size=0.15, test_size=0.15, random_seed=42
)
print(f"Train records: {len(X_train)} (Positive rate: {y_train.mean():.2%})")
print(f"Validation records: {len(X_val)} (Positive rate: {y_val.mean():.2%})")
print(f"Held-out Test records: {len(X_test)} (Positive rate: {y_test.mean():.2%})")

## 3. Fit Logistic Regression Pipeline on Training Set

In [ ]:
preprocessor = build_preprocessor(num_cols, cat_cols, bin_cols)
pipeline = build_classification_pipeline(preprocessor, class_weight="balanced", random_seed=42)
pipeline.fit(X_train, y_train)
print("Model pipeline fitted successfully on training split.")

## 4. Validation Set Threshold Tuning

In [ ]:
optimal_threshold, val_f1 = find_optimal_threshold(pipeline, X_val, y_val)
print(f"Optimal Decision Threshold: {optimal_threshold:.2f} (Validation F1: {val_f1:.4f})")

## 5. Held-out Test Set Evaluation

In [ ]:
metrics = evaluate_classifier(pipeline, X_test, y_test, threshold=optimal_threshold)
print(f"Test ROC-AUC: {metrics['roc_auc']:.4f}")
print(f"Test PR-AUC: {metrics['pr_auc']:.4f}")
print(f"Test Brier Score: {metrics['brier_score']:.4f}")
print("\nPerformance at Optimal Threshold:")
for k, v in metrics['optimal_threshold'].items():
    print(f"  {k}: {v}")

## 6. Interpretability: Top Feature Odds Ratios

In [ ]:
coef_df = extract_coefficients(pipeline)
print("Top 5 Positive Drivers (Increases Approval Odds):")
display(coef_df.head(5))

print("Top 5 Risk Drivers (Increases Rejection Odds):")
display(coef_df.tail(5).sort_values(by='coefficient', ascending=True))